In [0]:
from pyspark.sql.functions import current_timestamp, col
import os

BASE = "s3://mimic-iv-datasets/mimic-iv-dataset-3.1"
HOSP_PATH = f"{BASE}/hosp"
ICU_PATH = f"{BASE}/icu"

CATALOG = "workspace"
BRONZE = f"{CATALOG}.mimic_iv_3_1_bronze"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE}")

In [0]:
# This reads CSV files from the hosp/ folder and creates bronze tables

def create_bronze_table(source_path, table_name, schema_name):
    """
    Create a bronze table from CSV files in S3
    """
    try:
        # Read CSV with header inference
        df = (spark.read.format("csv")
                        .option("header", "true")
                        .option("inferSchema", "true")
                        .load(source_path))
        
        # Add metadata columns for bronze layer
        df_bronze = (df.withColumn("_ingestion_timestamp", current_timestamp())
                        .withColumn("_source_file", col("_metadata.file_path")))
        
        # Write to bronze table
        full_table_name = f"{schema_name}.{table_name}"
        (df_bronze.write.format("delta")
                        .mode("overwrite")
                        .option("overwriteSchema", "true")
                        .saveAsTable(full_table_name))
        
        row_count = df_bronze.count()
        print(f"Created {full_table_name} with {row_count:,} rows")
        return True
    except Exception as e:
        print(f"Failed to create {schema_name}.{table_name}: {str(e)}")
        return False

In [0]:
# Load all hosp tables to bronze layer
hosp_tables = [
    "admissions", "patients", "transfers", "diagnoses_icd", "d_icd_diagnoses",
    "procedures_icd", "d_icd_procedures", "prescriptions", "labevents", "d_labitems",
    "microbiologyevents", "poe", "poe_detail", "emar", "emar_detail",
    "pharmacy", "services"
]

print(f"Loading hosp tables from {HOSP_PATH}")

for hosp_table in hosp_tables:
    source_path = f"{HOSP_PATH}/{hosp_table}.csv*"
    create_bronze_table(source_path, f"hosp_{hosp_table}", BRONZE)

print("hosp bronze layer loading complete!")

In [0]:
# Load extra hosp tables missed to bronze layer
hosp_tables = [
    "d_hcpcs", "drgcodes", "omr", "hcpcsevents", "provider"
]

print(f"Loading hosp tables from {HOSP_PATH}")

for hosp_table in hosp_tables:
    source_path = f"{HOSP_PATH}/{hosp_table}.csv*"
    create_bronze_table(source_path, f"hosp_{hosp_table}", BRONZE)

print("hosp bronze layer loading complete!")

In [0]:
# Load all icu tables to bronze layer
icu_tables = [
    "icustays", "chartevents", "d_items", "datetimeevents", "inputevents",
    "outputevents", "procedureevents", "caregiver"
]

print(f"Loading icu tables from {ICU_PATH}")

for icu_table in icu_tables:
    source_path = f"{ICU_PATH}/{icu_table}.csv*"
    create_bronze_table(source_path, f"icu_{icu_table}", BRONZE)

print("icu bronze layer loading complete!")

In [0]:
# Load extra icu tables missed to bronze layer
icu_tables = [
    "ingredientevents"
]

print(f"Loading icu tables from {ICU_PATH}")

for icu_table in icu_tables:
    source_path = f"{ICU_PATH}/{icu_table}.csv*"
    create_bronze_table(source_path, f"icu_{icu_table}", BRONZE)

print("icu bronze layer loading complete!")